# Semana 11: Evaluación y diagnóstico

**Pregunta de trabajo.** Construir baseline, pipeline, CV, curva de aprendizaje y conteos por slice.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.model_selection import train_test_split, cross_val_score, learning_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

## Dataset local
Usamos `load_breast_cancer`, disponible sin internet.

In [2]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = data.target
X.shape, y.value_counts(normalize=True).sort_index()

((569, 30),
 target
 0    0.372583
 1    0.627417
 Name: proportion, dtype: float64)

## Funciones de evaluación

In [3]:
def make_splits(X, y, test_size=0.2, dev_size=0.25, random_state=42):
    X_train_dev, X_test, y_train_dev, y_test = train_test_split(
        X,
        y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )
    X_train, X_dev, y_train, y_dev = train_test_split(
        X_train_dev,
        y_train_dev,
        test_size=dev_size,
        stratify=y_train_dev,
        random_state=random_state,
    )
    return X_train, X_dev, X_test, y_train, y_dev, y_test


def metric_report(y_true, y_pred):
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
    }


def build_baseline(strategy="most_frequent"):
    return DummyClassifier(strategy=strategy)


def build_pipeline(C=1.0, random_state=42):
    return Pipeline(
        [
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    C=C,
                    max_iter=5000,
                    solver="liblinear",
                    random_state=random_state,
                ),
            ),
        ]
    )


def cross_validate_model(model, X, y, cv=5, scoring="f1"):
    scores = cross_val_score(model, X, y, cv=cv, scoring=scoring)
    return {"mean": float(np.mean(scores)), "std": float(np.std(scores)), "scores": scores}


def learning_curve_summary(model, X, y, train_sizes=None, cv=5, scoring="f1"):
    if train_sizes is None:
        train_sizes = np.array([0.3, 0.6, 1.0])
    sizes, train_scores, validation_scores = learning_curve(
        model,
        X,
        y,
        train_sizes=train_sizes,
        cv=cv,
        scoring=scoring,
    )
    return {
        "train_sizes": sizes,
        "train_scores_mean": train_scores.mean(axis=1),
        "validation_scores_mean": validation_scores.mean(axis=1),
    }


def slice_error_report(X, y_true, y_pred, feature, threshold):
    X_df = pd.DataFrame(X).copy()
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = X_df[feature].to_numpy() >= threshold
    rows = []
    for label, group_mask in [(f"{feature} < {threshold:.3g}", ~mask), (f"{feature} >= {threshold:.3g}", mask)]:
        yt = y_true[group_mask]
        yp = y_pred[group_mask]
        rows.append(
            {
                "slice": label,
                "n": int(group_mask.sum()),
                "f1": float(f1_score(yt, yp, zero_division=0)),
                "errors": int(np.sum(yt != yp)),
            }
        )
    return pd.DataFrame(rows)


def choose_best_model(results, metric="f1"):
    best_name = max(results, key=lambda name: results[name]["mean"])
    return best_name, float(results[best_name]["mean"])

## Particiones y baseline

In [4]:
X_train, X_dev, X_test, y_train, y_dev, y_test = make_splits(X, y, random_state=42)
baseline = build_baseline()
baseline.fit(X_train, y_train)
metric_report(y_dev, baseline.predict(X_dev))

{'accuracy': 0.6228070175438597,
 'precision': 0.6228070175438597,
 'recall': 1.0,
 'f1': 0.7675675675675676,
 'confusion_matrix': array([[ 0, 43],
        [ 0, 71]])}

## Pipeline y cross-validation

In [5]:
pipe = build_pipeline(C=1.0, random_state=42)
cv_summary = cross_validate_model(pipe, X_train, y_train, cv=5, scoring="f1")
cv_summary

{'mean': 0.98160613636721,
 'std': 0.005527129287256608,
 'scores': array([0.98850575, 0.97727273, 0.98823529, 0.97727273, 0.97674419])}

## Validación y error analysis

In [6]:
pipe.fit(X_train, y_train)
y_dev_pred = pipe.predict(X_dev)
metric_report(y_dev, y_dev_pred)

{'accuracy': 0.9824561403508771,
 'precision': 0.9859154929577465,
 'recall': 0.9859154929577465,
 'f1': 0.9859154929577465,
 'confusion_matrix': array([[42,  1],
        [ 1, 70]])}

In [7]:
threshold = float(X_dev["mean radius"].median())
slice_error_report(X_dev, y_dev, y_dev_pred, feature="mean radius", threshold=threshold)

,slice,n,f1,errors
0,mean radius < 13.4,57,0.991150,1
1,mean radius >= 13.4,57,0.965517,1


## Learning curve

In [8]:
learning_curve_summary(build_pipeline(C=1.0), X_train, y_train, train_sizes=[0.4, 0.7, 1.0], cv=3)

{'train_sizes': array([ 90, 158, 227]),
 'train_scores_mean': array([0.99376947, 0.98986044, 0.99067576]),
 'validation_scores_mean': array([0.97702728, 0.98156779, 0.97891697])}

## Cierre

Escriba una síntesis en forma: resultado → lectura → comprobación → límite.

In [9]:
candidate_results = {
    "logistic": cv_summary,
    "logistic_repeat": cross_validate_model(build_pipeline(C=0.2), X_train, y_train, cv=5, scoring="f1"),
}
conservative = pd.DataFrame([
    {"model": name, "cv_mean": value["mean"], "cv_std": value["std"], "score": value["mean"] - value["std"]}
    for name, value in candidate_results.items()
]).sort_values("score", ascending=False)
conservative

,model,cv_mean,cv_std,score
1,logistic_repeat,0.986098,0.008576,0.977521
0,logistic,0.981606,0.005527,0.976079
